In [35]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb langchain_text_splitters 

## RAG

In [102]:
from langchain_core.documents import Document
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders.pdf import PyPDFLoader
import os 
import chromadb
from sklearn.metrics.pairwise import cosine_similarity
import uuid
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [37]:
# Loading Text Data

loader = TextLoader("X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\RAG\\DATA\\Py.txt",encoding='utf-8')
documents = loader.load()

# text 

In [38]:
# Loading PDF Data

pdf = PyPDFLoader("X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\RAG\\DATA\\PDF\\research2.pdf")
documents = pdf.load()

# pdf_doc

In [32]:
# Ingestion pipeline

def LoadPDF():
    folder_path = "X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\RAG\\DATA\\PDF"
    num_doc = 0
    all_doc = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith('.pdf'):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            all_doc.extend(doc)
            num_doc += 1

    print(f"Total PDF: {num_doc}\n")
    print(f"Total PDF Pages: {len(all_doc)}")
    return all_doc

In [33]:
pdf_docs = LoadPDF()

Total PDF: 2

Total PDF Pages: 32


In [66]:
def splitDoc(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    chunked_docx = text_splitter.split_documents(documents)
    return chunked_docx

In [67]:
chunks = splitDoc(pdf_docs)
print(f'Length of chunks: {len(chunks)}')

Length of chunks: 321


In [68]:
# Embedding

In [90]:
class EmbeddingManager:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def generate_embedding(self, text):
        embedding = self.model.encode(text, show_progress_bar=True)
        print(f"Embedding Shape: {embedding.shape}")
        return embedding

In [91]:
embedding_manager = EmbeddingManager()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [92]:
# vector store

In [99]:
class VectorStore:
    def __init__(self, persist_dir='X:\\\\VScode\\\\Artificial_intelligence_n_Machine_learning\\\\Deep_Learning\\\\RAG\\\\DATA\\\\vector_store', collection_name='pdf_docx'):
        self.persist_dir = persist_dir
        self.collection_name = collection_name  
        self.client = None
        self.collection = None
        
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_dir, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_dir)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "vector store collection for PDF embedding in RAG"    
            }
        )
        print("Initialized the vector store with collection =", self.collection_name)
        print('Docs in collection:', self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of Embeddings")

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embed) in enumerate(zip(documents, embeddings)):
            doc_id = f'doc_{uuid.uuid4()}'
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            all_metadata.append(metadata)
            documents_content.append(doc.page_content)
            embeddings_list.append(embed.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        print(f"Successfully added {len(documents)} documents to the vector store.")
        print('Docs in collection:', self.collection.count())

In [100]:
vector_store = VectorStore()

Initialized the vector store with collection = pdf_docx
Docs in collection: 0


In [101]:
texts = [doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embedding(texts)
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embedding Shape: (321, 384)
Successfully added 321 documents to the vector store.
Docs in collection: 321


In [109]:
class RAGRetriver:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrive(self, query, top_k=5, score_threshold=0.0):
        query_embedding = self.embedding_manager.generate_embedding(query)

        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(), 
            n_results=top_k
        )
        
        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            ids = results['ids'][0]
            metadatas = results['metadatas'][0]
            documents = results['documents'][0]
            distances = results['distances'][0]

            for i, (doc_id, metadata, doc_content, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': doc_id,
                        'document': doc_content,
                        'metadata': metadata,
                        'distance': distance,
                        'similarity': similarity_score,
                        'rank': i + 1
                    })
        else:
            print("No Documents found!")
            
        return retrieved_docs

In [110]:
rag_retriever = RAGRetriver(embedding_manager, vector_store)

In [112]:
rag_retriever.retrive("what is Embedding")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (384,)


[{'id': 'doc_d982375f-728f-4119-ae56-5f8bb161d951',
  'document': 'linear transformation, similar to [24]. In the embedding layers, we multiply those weights by√dmodel.\n3.5 Positional Encoding\nSince our model contains no recurrence and no convolution, in order for the model to make use of the\norder of the sequence, we must inject some information about the relative or absolute position of the\ntokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the\n5\nrajatb1008@gmail.com',
  'metadata': {'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'title': 'Attention is All you Need',
   'type': 'Conference Proceedings',
   'source': 'X:\\VScode\\Artificial_intelligence_n_Machine_learning\\Deep_Learning\\RAG\\DATA\\PDF\\NIPS-2017-attention-is-all-you-need-Paper.pdf',
   'eventtype': 'Poster',
   'book': 'Advances in Neural Information Processing Systems 30',
   'author': 'Ashis